# Lesson 3.1: Loading and Cleaning Data

**🎬 Video:** [Lesson 3.1: Loading and Cleaning Data](#)

## Overview

In this lesson you will learn how to bring data into Python and prepare it for analysis. We will be using a powerful library called pandas (<img src="https://pandas.pydata.org/static/img/pandas.svg" style="height: 24px; vertical-align: middle;">). pandas is designed for cleaning and analyzing data and is widely used in data science. By the end you will have a clean, properly typed DataFrame saved to disk, ready for visualization in the next lesson.

**What you will cover:**
- Import data libraries
- Load a CSV file into a pandas DataFrame
- Understand and fix data types
- Clean text data with regular expressions
- Save your cleaned data for reuse

---

## 📖 1 Follow Along — Loading Libraries

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

Most projects in Python will require libraries - collections of code that help you achieve particular tasks. These packages aren't loaded by default because loading a package that isn't used is a waste of memory and because different packages can cause conflicts. For this lesson we will use **pandas** for working with tabular data.

Packages are always imported at the top of a script even if their functions are not called till much later. This is a convention that helps organize scripts for readability. 

Always make sure you load in all libraries at the beginning of a session.

- Click the <img src="../lesson_assets/images/jupyter/run.svg" style="height: 14pt; vertical-align: middle;"> button below to load libraries

In [ ]:
# Import the required libraries for data analysis and visualization
import pandas as pd          # For working with data tables (DataFrames)


---

## 1 Getting Data

The most basic step in any data science project is actually importing the data. The pandas library makes this very easy if you are working with tabular data (data organized in rows and columns, like a spreadsheet). 

We will be working with a CSV (Comma-Separated Values) file containing Reddit posts from the r/JMU subreddit. CSV files are one of the most common formats for storing data because they can be opened by almost any program.

We can load the Reddit CSV file using pandas' `.read_csv()` function.




### 1.1 Loading a local file

To load the CSV file locally, we use the command `pd.read_csv("filename.csv")`. The `pd` is our shorthand for pandas, and `read_csv()` is the function that reads CSV files.

In [ ]:
# Load the CSV file and display it (but don't save it to a variable yet)
pd.read_csv("../data/JMU/JMU_raw.csv")

> 👉 **Note:** *All we did here is load the file into the output screen of Jupyter. We still have to store it as a variable if we want to use it.*

We can store it by having the result of the function equal a variable.

In [ ]:
# Store the CSV file as a variable called 'df' (short for DataFrame)
df = pd.read_csv("../data/JMU/JMU_raw.csv")
# Display the first 5 rows to see what our data looks like
df.head()

The CSV file has now been stored as a `DataFrame`. A DataFrame is a lot like a spreadsheet with columns and rows, but it has some features to optimize it for data science analysis.

We can show the content of the DataFrame by typing `df`. The `.head()` method shows us the first 5 rows, which is useful for getting a quick look at our data.

> 👉 **Note:** *You do not have to use the name `df` — this is just a common convention that stands for "DataFrame".*

---

## 2 Setting Data Types

Before you start doing anything with your data, you want to make sure you get the DataFrame cleaned up. This will make working with the data easier. 


### 2.1 Data Types

Part of the power of pandas is that it assumes each column contains the same type of data (all numbers, all text, all dates, etc.). This allows it to make calculations much faster. 

However, when you import from a CSV file, pandas is not always able to automatically determine what type of data is in each column. We can check what pandas decided by using the `.dtypes` property (note: no parentheses needed).

In [ ]:
# Check what data types Pandas assigned to each column
df.dtypes

pandas was able to figure out that the **score** column contains integers, but it had trouble with the other columns. It saved these as generic `objects`, which means they could be strings, numbers, or other types of data formats. 

This can lead to problems later, so pandas recommends converting text columns to their special string format called `StringDtype`. This makes for better storage and processing. The code below converts each column to its proper data type:

**Syntax Explanation:** To change a column's data type, we use this pattern:
- `df['column_name']` - selects the column
- `.astype('new_type')` - converts it to the new data type
- `df['column_name'] = ...` - saves the converted column back to the DataFrame

In [ ]:
# Convert each column to its proper data type
df["type"] = df["type"].astype("category")        # Post type (post/comment) as category
df["text"] = df["text"].astype(pd.StringDtype())    # Post content as string  
df["date"] = pd.to_datetime(df["date"])             # Date as datetime
df["score"] = pd.to_numeric(df["score"])             # Score as number
# Check our new data types
df.dtypes


Great! Now our columns have proper data types. This is basic housekeeping you should do whenever working with CSV files. While it might seem tedious, it prevents many problems later in your analysis.

---

## 3 Cleaning the Data

Once the DataFrame is in order, you will want to clean up some of the data. This is usually a recursive process. That is, you usually only figure out that there is an issue with the data when you start working with it. As you keep finding issues, you want to clean these issues earlier in your code, rather than when you run into them.

### 3.1 Normalizing punctuation

Text copied from websites and word processors often contains "smart" punctuation that makes the document look nicer like: curly quotes (`‘` `’` `“` `”`), em dashes (`—`), ellipses (`…`), and bullet points (`•`). These are valid Unicode characters and look perfectly fine on screen, but they can appear as garbled sequences like `â€™` or `â€œ` when a CSV is opened in Excel or Google Sheets, which sometimes default to a different text encoding.

The safest fix is to replace these characters with their plain ASCII equivalents before saving. ASCII is the most basic version of text encoding that covers only the characters on the keyboard. This keeps the CSV readable in any program without changing the meaning of the text.

The code below loops through the `text` columns and swaps each smart character for its plain version:


In [ ]:
# Map "smart" Unicode punctuation to plain ASCII equivalents
punctuation_map = {
    "\u2018": "'",    # left single quote   '
    "\u2019": "'",    # right single quote  '
    "\u201c": '"',    # left double quote   "
    "\u201d": '"',    # right double quote  "
    "\u2013": "-",    # en dash             –
    "\u2014": "-",    # em dash             —
    "\u2026": "...",  # ellipsis            …
    "\u2022": "-",    # bullet point        •
    "\u00ad": "",     # soft hyphen (invisible — just remove it)
}

for col in ["text"]:
    for char, replacement in punctuation_map.items():
        df[col] = df[col].str.replace(char, replacement, regex=False)

# Confirm no smart punctuation remains
remaining = df["text"].str.contains("|".join(punctuation_map.keys()), regex=True, na=False).sum()
print(f"Rows with smart punctuation remaining: {remaining}")


### 3.2 Removing special characters


When working with text data, a common problem is that special characters like emojis can interfere with analysis. These need to be removed because they might cause issues when using tools for **sentiment analysis** or **text processing**. You'll get better results if your data is normalized.

- **Sentiment analysis** - tools work by matching words against a dictionary of known positive and negative terms — emojis are not words, so they are either ignored or misread as garbled characters, causing the tool to miss emotional signals entirely.
- **Text processing** - tools such as tokenizers and vectorizers split text into individual words; an emoji embedded in a word (e.g. `great😊day`) prevents the split from happening correctly, corrupting the tokens that follow.

The code snippet below creates a search pattern for emojis and then finds any emoji in the `text` column by going row by row looking for emojis. It then shows all rows with emojis. 

In [ ]:
# Define a regex (regular expression) pattern that matches common emoji characters
# This pattern covers most emoji Unicode ranges
emoji_pattern = r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]"

# Find all rows where the 'text' column contains emojis
rows_with_emojis = df[df["text"].str.contains(emoji_pattern, regex=True, na=False)]

# Show just the text column of these rows to see the emojis
rows_with_emojis[["text"]]

> 📊 **Output:** In the result above, you can see there are many rows with emojis. 
 
We could manually remove each emoji by editing each individual cell, but that would be extremely time-consuming. Instead, we'll use pandas to automatically find and remove all emojis by replacing them with empty text. This is essentially like doing a **find + replace** in a document, but instead of replacing only one word or character, many characters are being replaced at the same time. 

The code to do this is:
```python
df['text'] = df['text'].str.replace(emoji_pattern, '', regex=True)
```
This reads as: in the `'text'` column, find every emoji and swap it with nothing (`''`). Replacing with an empty string is effectively the same as deleting — there is no separate "delete" function because the result is identical.

In [ ]:
# This pattern covers most emoji Unicode ranges
emoji_pattern = r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]"

# Remove all emojis from the text column by replacing them with empty strings
df["text"] = df["text"].str.replace(emoji_pattern, "", regex=True)

# Check if our emoji removal worked by searching for emojis again
rows_with_emojis_cleaned = df[df["text"].str.contains(emoji_pattern, regex=True, na=False)]

# Display the results (should be empty or much fewer rows)
rows_with_emojis_cleaned[["text"]]

> 👉 **Note:** *Removing emojis can be a good practice when working with sentiment analysis, because computers have a hard time understanding how emojis are being used. That said, you are also removing nuance from of the data.*

Consider this sentence:

    `Number 1 school for cases in VA. Roll dukes 😝`

What does it mean and how is the emoji being used? Is this a positive or negative emotion? 

As you work with a DataFrame there are always other formatting quirks in the data you will want to take care of. It makes little sense to try to clean everything in advance and hope for the best. Likely, you'll find problems as you go and then make the fixes part of the cleaning process.

---

## 4 Saving Your Work

Great work! We now have a clean DataFrame that we can use for further analysis. Whenever you run code in Jupyter, the notebook automatically stores the values of the variables. When you restart, these values are deleted. It is a good practice to save the DataFrame variable once you have performed a lot of operations on it. This prevents us from having to run the above code every time. 
There are a couple of ways we can save this as a `csv` file using `to_csv()` or a `pickle` file or `to_pickle()`. There are several advantages and disadvantages to each. 

##### CSV

**Advantages**

- Easy to share
- Can open on most computers

**Disadvantages**

- They tend to be big
- Read and write times can be slow
- Unless you specifically indicate the column types, the `dtype` will get lost. This is a huge pain.

##### Pickle

**Advantages**

- Smaller
- Faster
- Keeps dtypes

**Disadvantages**

- Requires Python to open


Saving to either format is incredibly simple, and you could equally well save to both formats to anticipate both situations.

In [ ]:
df.to_csv("../data/JMU/JMU_raw_cleaned.csv")
df.to_pickle("../data/JMU/JMU_raw_cleaned.pickle")

The code above saves the DataFrame to your `data/` folder in both formats. In the next lesson you will load this file and build interactive visualizations from it.

> 👉 **Note:** *As a rule, most files are saved in both formats for the purposes of this lesson. This way you have one version for data processing `*.pickle` and one version you can click on and see. Generally though you would only have pickle files and maybe CSV files for the initial input and final output.*

---

## Lesson Summary

Here is what you covered in this lesson:

### Part 1: Loading Data
- **`pd.read_csv()`** — load a CSV file into a DataFrame
- **`df.head()`** — preview the first 5 rows


### Part 2: Type Conversion
- **`pd.to_datetime()`** — convert a text column into a proper datetime object
- **`.astype('category')`** — convert a column to a memory-efficient categorical type
- **`pd.StringDtype()`** — convert a column to a pandas special string type

### Part 3: Cleaning Data
- **`.str.replace()`** with a character map — normalize smart punctuation to plain ASCII
- **`.str.contains()`** with regex — find rows matching a pattern
- **`.str.replace()`** with regex — strip unwanted characters from text
- **`r'...'` raw strings** — define regex patterns without double-escaping backslashes

### Part 4: Saving Data
- **`df.to_pickle()`** — save a DataFrame with all data types preserved
- **`df.to_csv()`** — save a DataFrame as a portable CSV file

---

In the next lesson you will load this cleaned data and build interactive visualizations with Plotly.

➡️ **Next:** [Lesson 3.2 — Visualizing Data with Plotly](lesson_3_2_plotly_charts.ipynb)
